给定字符序列："ababc"
词汇表：{ 'a', 'b', 'c' }

一阶马尔可夫模型，使用加1平滑（拉普拉斯平滑）：

条件概率公式：
p(x_t | x_{t-1}) = ( count(x_{t-1}, x_t) + 1 ) / ( count(x_{t-1}) + |V| )

其中 |V| = 3（词汇表大小）

统计转移频次（从序列 "ababc" 中统计相邻对）：
序列位置： a b a b c
相邻对：
(a -> b) : 出现 2 次（位置1-2，位置3-4）
(b -> a) : 出现 1 次（位置2-3）
(b -> c) : 出现 1 次（位置4-5）

因此：
count(b) = 从 b 出发的总次数 = b->a 次数 + b->c 次数 = 1 + 1 = 2

计算：
1. p(a' | b') = ( count(b->a) + 1 ) / ( count(b) + 3 )
              = ( 1 + 1 ) / ( 2 + 3 )
              = 2 / 5
              = 0.4

2. p(c' | b') = ( count(b->c) + 1 ) / ( count(b) + 3 )
              = ( 1 + 1 ) / ( 2 + 3 )
              = 2 / 5
              = 0.4

答案：
p(a' | b') = 0.4
p(c' | b') = 0.4

In [1]:
import re

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表，生成特征序列和标签序列（用于自回归语言模型）
    
    参数：
        text (str): 输入文本
        n (int): 滑动窗口大小（特征长度）
    
    返回：
        vocab (dict): 词汇表，词到整数ID的映射（按频率降序，从0开始）
        features (list): 特征序列，每个特征是一个长度为n的词列表
        labels (list): 标签序列，每个标签是下一个词
    """
    # 1. 转小写，去除非字母和空格
    text_lower = text.lower()
    # 只保留字母和空格
    text_clean = re.sub(r'[^a-z\s]', '', text_lower)
    
    # 2. 按空格分词，并去掉空字符串
    words = text_clean.split()
    
    # 3. 构建词汇表（按频率排序，从高到低）
    word_freq = {}
    for word in words:
        word_freq[word] = word_freq.get(word, 0) + 1
    
    # 按频率降序排序，频率相同则按字母顺序
    sorted_words = sorted(word_freq.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 生成特征和标签（滑动窗口）
    features = []
    labels = []
    
    for i in range(len(words) - n):
        # 取当前窗口的n个词作为特征
        feature = words[i:i+n]
        # 下一个词作为标签
        label = words[i+n]
        features.append(feature)
        labels.append(label)
    
    return vocab, features, labels


# 测试示例
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, features, labels = preprocess_text(text, n)
    
    print("词汇表:", vocab)
    print("特征序列:", features)
    print("标签序列:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征序列: [['the', 'time']]
标签序列: ['machine']


线性RNN定义：
h_t = W_hh * h_{t-1} + W_hx * x_t
o_t = W_oh * h_t

损失函数：
L = (1/2) * sum_{t=1}^T (o_t - y_t)^2

令 delta_t = dL / do_t = o_t - y_t （标量，因为输出是标量）
令 gamma_t = dL / dh_t

根据链式法则：
gamma_t = dL / dh_t
        = (dL / do_t) * (do_t / dh_t) + (dL / dh_{t+1}) * (dh_{t+1} / dh_t)
        = delta_t * W_oh^T + gamma_{t+1} * W_hh^T

其中 gamma_{T+1} = 0（在最后一个时间步之后没有隐藏状态）

展开 gamma_t：
gamma_t = delta_t * W_oh^T + gamma_{t+1} * W_hh^T
        = delta_t * W_oh^T + (delta_{t+1} * W_oh^T + gamma_{t+2} * W_hh^T) * W_hh^T
        = delta_t * W_oh^T + delta_{t+1} * W_oh^T * W_hh^T + gamma_{t+2} * (W_hh^T)^2
        = sum_{k=t}^T delta_k * W_oh^T * (W_hh^T)^{k-t}

因此，损失对 W_hh 的梯度：
dL / dW_hh = sum_{t=1}^T (dL / dh_t) * (dh_t / dW_hh)
           = sum_{t=1}^T gamma_t * h_{t-1}^T
           = sum_{t=1}^T [sum_{k=t}^T delta_k * W_oh^T * (W_hh^T)^{k-t}] * h_{t-1}^T

梯度中的关键项是 (W_hh^T)^{k-t}，即 W_hh 的幂次。

梯度消失或爆炸的条件：
- 如果 W_hh 的谱半径（最大特征值的绝对值）> 1，则随着 (k-t) 增大，(W_hh^T)^{k-t} 呈指数增长，导致梯度爆炸。
- 如果 W_hh 的谱半径 < 1，则随着 (k-t) 增大，(W_hh^T)^{k-t} 呈指数衰减，导致梯度消失。
- 如果 W_hh 的谱半径 = 1，梯度既不会消失也不会爆炸（理想情况）。

总结：梯度消失/爆炸取决于 W_hh 的特征值是否在单位圆内。谱半径 < 1 导致梯度消失，> 1 导致梯度爆炸。

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元前向传播
    
    参数：
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一时刻隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏权重，形状 (input_size, hidden_size)
        W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
        b_h: 隐藏层偏置，形状 (hidden_size,)
    
    返回：
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播 (x_t, h_prev, W_hx, W_hh, b_h, h_t, pre_activation)
    """
    # 计算线性组合
    pre_activation = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    # tanh激活
    h_t = np.tanh(pre_activation)
    
    cache = (x_t, h_prev, W_hx, W_hh, b_h, h_t, pre_activation)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN单元反向传播（单步）
    
    参数：
        dh_next: 上游梯度，即 dL/dh_t，形状 (batch_size, hidden_size)
        cache: 前向传播缓存的元组
    
    返回：
        dx_t: 损失对输入的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对上一隐藏状态的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对输入权重的梯度，形状 (input_size, hidden_size)
        dW_hh: 损失对隐藏权重的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对偏置的梯度，形状 (hidden_size,)
    """
    x_t, h_prev, W_hx, W_hh, b_h, h_t, pre_activation = cache
    
    batch_size = x_t.shape[0]
    
    # tanh的导数: d(tanh(z))/dz = 1 - tanh(z)^2
    dtanh = 1 - h_t ** 2  # 形状 (batch_size, hidden_size)
    
    # 上游梯度乘以tanh导数
    dh = dh_next * dtanh  # 形状 (batch_size, hidden_size)
    
    # 计算各个梯度
    # dW_hx = dx_t^T * dh
    dx_t = np.dot(dh, W_hx.T)  # 形状 (batch_size, input_size)
    
    # dh_prev = dh * W_hh^T
    dh_prev = np.dot(dh, W_hh.T)  # 形状 (batch_size, hidden_size)
    
    # dW_hx = x_t^T * dh
    dW_hx = np.dot(x_t.T, dh)  # 形状 (input_size, hidden_size)
    
    # dW_hh = h_prev^T * dh
    dW_hh = np.dot(h_prev.T, dh)  # 形状 (hidden_size, hidden_size)
    
    # db_h = sum(dh, axis=0)
    db_h = np.sum(dh, axis=0)  # 形状 (hidden_size,)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试代码
if __name__ == "__main__":
    # 设置随机种子以便复现
    np.random.seed(42)
    
    # 参数设置
    batch_size = 3
    input_size = 4
    hidden_size = 5
    
    # 随机初始化输入、隐藏状态和权重
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("前向传播结果:")
    print(f"h_t 形状: {h_t.shape}")
    print(f"h_t 前5个值: {h_t[0, :5]}")
    
    # 反向传播（假设上游梯度为随机值）
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)
    
    print("\n反向传播结果:")
    print(f"dx_t 形状: {dx_t.shape}")
    print(f"dh_prev 形状: {dh_prev.shape}")
    print(f"dW_hx 形状: {dW_hx.shape}")
    print(f"dW_hh 形状: {dW_hh.shape}")
    print(f"db_h 形状: {db_h.shape}")
    
    # 数值梯度验证（可选）
    print("\n执行梯度检查...")
    eps = 1e-6
    
    # 检查 dW_hh 的数值梯度
    W_hh_plus = W_hh.copy()
    W_hh_plus[0, 0] += eps
    h_t_plus, _ = rnn_cell_forward(x_t, h_prev, W_hx, W_hh_plus, b_h)
    
    W_hh_minus = W_hh.copy()
    W_hh_minus[0, 0] -= eps
    h_t_minus, _ = rnn_cell_forward(x_t, h_prev, W_hx, W_hh_minus, b_h)
    
    # 假设损失函数为 L = sum(h_t * dh_next)（线性近似）
    L_plus = np.sum(h_t_plus * dh_next)
    L_minus = np.sum(h_t_minus * dh_next)
    numerical_grad = (L_plus - L_minus) / (2 * eps)
    
    print(f"dW_hh[0,0] 数值梯度: {numerical_grad:.6f}")
    print(f"dW_hh[0,0] 反向传播梯度: {dW_hh[0,0]:.6f}")
    print(f"相对误差: {abs(numerical_grad - dW_hh[0,0]) / (abs(numerical_grad) + abs(dW_hh[0,0]) + 1e-8):.6e}")

前向传播结果:
h_t 形状: (3, 5)
h_t 前5个值: [ 0.37757047 -0.85065863 -0.99999996 -0.95887056  0.60632966]

反向传播结果:
dx_t 形状: (3, 4)
dh_prev 形状: (3, 5)
dW_hx 形状: (4, 5)
dW_hh 形状: (5, 5)
db_h 形状: (5,)

执行梯度检查...
dW_hh[0,0] 数值梯度: -0.046423
dW_hh[0,0] 反向传播梯度: -0.046423
相对误差: 6.598333e-10


深度双向RNN参数分析：

给定：
- L 层
- 每层隐藏单元数 H
- 输入维度 D
- 输出维度 O（仅考虑最后输出层）

每层双向RNN包含：
1. 前向RNN层
2. 反向RNN层

对于每一层 l（从1到L）：

第1层（l=1）：
- 前向RNN：输入维度为 D（原始输入），隐藏维度为 H
  权重 W_hx：D * H
  权重 W_hh：H * H
  偏置 b_h：H
  参数小计：D*H + H*H + H

- 反向RNN：输入维度为 D（原始输入），隐藏维度为 H
  权重 W_hx：D * H
  权重 W_hh：H * H
  偏置 b_h：H
  参数小计：D*H + H*H + H

第1层参数总计：2 * (D*H + H*H + H)

第l层（l >= 2）：
- 前向RNN：输入维度为 2*H（因为上一层的双向输出拼接），隐藏维度为 H
  权重 W_hx：(2*H) * H
  权重 W_hh：H * H
  偏置 b_h：H
  参数小计：2*H*H + H*H + H = 3*H*H + H

- 反向RNN：输入维度为 2*H（因为上一层的双向输出拼接），隐藏维度为 H
  权重 W_hx：(2*H) * H
  权重 W_hh：H * H
  偏置 b_h：H
  参数小计：2*H*H + H*H + H = 3*H*H + H

第l层（l >= 2）参数总计：2 * (3*H*H + H) = 6*H*H + 2*H

总参数数量：
Total = 第1层参数 + 第2到L层参数
      = 2*(D*H + H*H + H) + (L-1)*(6*H*H + 2*H)

化简：
Total = 2*D*H + 2*H*H + 2*H + (L-1)*(6*H*H + 2*H)
      = 2*D*H + 2*H*H + 2*H + 6*(L-1)*H*H + 2*(L-1)*H
      = 2*D*H + [2 + 6*(L-1)]*H*H + [2 + 2*(L-1)]*H
      = 2*D*H + (6*L - 4)*H*H + 2*L*H

最终表达式：
Total Parameters = 2*D*H + (6*L - 4)*H^2 + 2*L*H

其中：
- 第一项 2*D*H：第1层输入到隐藏的权重
- 第二项 (6*L - 4)*H^2：所有层的隐藏到隐藏权重（包括前向和反向）
- 第三项 2*L*H：所有层的偏置

如果省略偏置项，则：
Total Parameters = 2*D*H + (6*L - 4)*H^2

In [5]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    返回每个时间步拼接后的前向和反向隐藏状态，以及最终时间步的拼接隐藏状态
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1, rnn_type='rnn'):
        """
        参数：
            input_dim: 输入维度
            hidden_dim: 隐藏层维度（每个方向）
            num_layers: RNN层数
            rnn_type: RNN类型，可选 'rnn', 'lstm', 'gru'
        """
        super(BidirectionalRNNEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.rnn_type = rnn_type
        
        # 根据类型选择RNN
        if rnn_type == 'rnn':
            self.rnn = nn.RNN(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,  # 保持 (seq_len, batch, input_dim)
                bidirectional=True
            )
        elif rnn_type == 'lstm':
            self.rnn = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,
                bidirectional=True
            )
        elif rnn_type == 'gru':
            self.rnn = nn.GRU(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,
                bidirectional=True
            )
        else:
            raise ValueError("rnn_type must be 'rnn', 'lstm', or 'gru'")
    
    def forward(self, X):
        """
        前向传播
        
        参数：
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回：
            outputs: 每个时间步拼接后的隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        # 运行双向RNN
        output, hidden = self.rnn(X)
        # output 形状: (seq_len, batch, 2*hidden_dim)
        
        # 获取最终时间步的隐藏状态
        # 对于双向RNN，需要分别获取前向和反向的最后一个时间步
        if self.rnn_type == 'lstm':
            # LSTM返回 (h_n, c_n)，我们使用h_n
            h_n = hidden[0]  # 形状: (2*num_layers, batch, hidden_dim)
        else:
            h_n = hidden  # 形状: (2*num_layers, batch, hidden_dim)
        
        # 获取最后一层的前向和反向隐藏状态
        # 前向：最后一层的前向 (索引为 2*num_layers - 2)
        # 反向：最后一层的反向 (索引为 2*num_layers - 1)
        forward_last = h_n[-2, :, :]  # 形状: (batch, hidden_dim)
        backward_last = h_n[-1, :, :]  # 形状: (batch, hidden_dim)
        
        # 拼接得到最终状态
        final_state = torch.cat([forward_last, backward_last], dim=-1)  # (batch, 2*hidden_dim)
        
        return output, final_state


# 修正后的多层手动实现
class BidirectionalRNNEncoderManual(nn.Module):
    """
    手动实现的双向RNN编码器（使用RNNCell）
    用于理解双向RNN的工作原理
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoderManual, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 为每一层创建前向和反向的RNN单元
        self.forward_cells = nn.ModuleList()
        self.backward_cells = nn.ModuleList()
        
        for layer in range(num_layers):
            # 第一层输入维度为 input_dim
            # 其他层输入维度为 2*hidden_dim（因为要拼接前向和反向输出作为下一层的输入）
            if layer == 0:
                layer_input_dim = input_dim
            else:
                layer_input_dim = 2 * hidden_dim
            
            # 前向单元
            self.forward_cells.append(
                nn.RNNCell(layer_input_dim, hidden_dim)
            )
            # 反向单元
            self.backward_cells.append(
                nn.RNNCell(layer_input_dim, hidden_dim)
            )
    
    def forward(self, X):
        """
        前向传播
        
        参数：
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回：
            outputs: 每个时间步拼接后的隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        seq_len, batch_size, _ = X.shape
        device = X.device
        
        # 存储所有时间步的输出
        outputs = []
        
        # 初始化每一层的隐藏状态
        forward_hidden = [torch.zeros(batch_size, self.hidden_dim).to(device) 
                         for _ in range(self.num_layers)]
        backward_hidden = [torch.zeros(batch_size, self.hidden_dim).to(device) 
                          for _ in range(self.num_layers)]
        
        # 第一步：前向传播，存储所有层的输出
        forward_outputs = []  # 存储每个时间步所有层的输出
        for t in range(seq_len):
            x_t = X[t]  # (batch, input_dim)
            layer_outputs = []
            
            for layer in range(self.num_layers):
                if layer == 0:
                    # 第一层使用原始输入
                    f_input = x_t
                else:
                    # 其他层使用前一层的输出（拼接前向和反向输出）
                    # 注意：这里需要使用前一时间步的拼接输出
                    if t == 0:
                        # 第一个时间步，没有前一层的输出，使用零向量
                        prev_forward = torch.zeros(batch_size, self.hidden_dim).to(device)
                        prev_backward = torch.zeros(batch_size, self.hidden_dim).to(device)
                    else:
                        prev_forward = forward_outputs[t-1][layer-1]
                        prev_backward = backward_outputs[t-1][layer-1] if backward_outputs[t-1] is not None else torch.zeros(batch_size, self.hidden_dim).to(device)
                    
                    f_input = torch.cat([prev_forward, prev_backward], dim=-1)
                
                forward_hidden[layer] = self.forward_cells[layer](f_input, forward_hidden[layer])
                layer_outputs.append(forward_hidden[layer])
            
            forward_outputs.append(layer_outputs)
        
        # 第二步：反向传播，存储所有层的输出
        backward_outputs = [None] * seq_len
        for t in range(seq_len - 1, -1, -1):
            x_t = X[t]  # (batch, input_dim)
            layer_outputs = []
            
            for layer in range(self.num_layers):
                if layer == 0:
                    # 第一层使用原始输入
                    b_input = x_t
                else:
                    # 其他层使用前一层的反向输出（来自下一时间步）
                    if t == seq_len - 1:
                        # 最后一个时间步，没有下一时间步的输出，使用零向量
                        next_forward = torch.zeros(batch_size, self.hidden_dim).to(device)
                        next_backward = torch.zeros(batch_size, self.hidden_dim).to(device)
                    else:
                        next_forward = forward_outputs[t+1][layer-1]
                        next_backward = backward_outputs[t+1][layer-1]
                    
                    b_input = torch.cat([next_forward, next_backward], dim=-1)
                
                backward_hidden[layer] = self.backward_cells[layer](b_input, backward_hidden[layer])
                layer_outputs.append(backward_hidden[layer])
            
            backward_outputs[t] = layer_outputs
        
        # 第三步：拼接每个时间步的前向和反向输出（只使用最后一层）
        for t in range(seq_len):
            # 获取最后一层的输出
            forward_last = forward_outputs[t][-1]  # (batch, hidden_dim)
            backward_last = backward_outputs[t][-1]  # (batch, hidden_dim)
            combined = torch.cat([forward_last, backward_last], dim=-1)
            outputs.append(combined)
        
        # 转换为张量
        outputs = torch.stack(outputs, dim=0)  # (seq_len, batch, 2*hidden_dim)
        
        # 最终状态：最后一个时间步的拼接
        final_state = outputs[-1]  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# 更简单的单层双向RNN手动实现（用于演示）
class SimpleBidirectionalRNN(nn.Module):
    """
    简化的单层双向RNN手动实现
    """
    def __init__(self, input_dim, hidden_dim):
        super(SimpleBidirectionalRNN, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # 前向RNN单元
        self.forward_rnn = nn.RNNCell(input_dim, hidden_dim)
        # 反向RNN单元
        self.backward_rnn = nn.RNNCell(input_dim, hidden_dim)
    
    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        """
        seq_len, batch_size, _ = X.shape
        device = X.device
        
        # 初始化隐藏状态
        h_forward = torch.zeros(batch_size, self.hidden_dim).to(device)
        h_backward = torch.zeros(batch_size, self.hidden_dim).to(device)
        
        # 前向传播
        forward_outputs = []
        for t in range(seq_len):
            h_forward = self.forward_rnn(X[t], h_forward)
            forward_outputs.append(h_forward)
        
        # 反向传播
        backward_outputs = [None] * seq_len
        for t in range(seq_len - 1, -1, -1):
            h_backward = self.backward_rnn(X[t], h_backward)
            backward_outputs[t] = h_backward
        
        # 拼接输出
        outputs = []
        for t in range(seq_len):
            combined = torch.cat([forward_outputs[t], backward_outputs[t]], dim=-1)
            outputs.append(combined)
        
        outputs = torch.stack(outputs, dim=0)
        final_state = outputs[-1]
        
        return outputs, final_state


# 测试代码
if __name__ == "__main__":
    # 参数设置
    seq_len = 5
    batch_size = 3
    input_dim = 10
    hidden_dim = 8
    num_layers = 2
    
    # 创建随机输入
    X = torch.randn(seq_len, batch_size, input_dim)
    
    # 测试PyTorch内置的双向RNN
    print("=" * 50)
    print("测试 PyTorch 内置双向RNN:")
    print("=" * 50)
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers, rnn_type='rnn')
    outputs, final_state = encoder(X)
    
    print(f"输入形状: {X.shape}")
    print(f"输出形状: {outputs.shape}")  # 期望: (seq_len, batch, 2*hidden_dim)
    print(f"最终状态形状: {final_state.shape}")  # 期望: (batch, 2*hidden_dim)
    print(f"输出[0,0,:5]: {outputs[0, 0, :5].detach().numpy()}")
    print(f"最终状态[0,:5]: {final_state[0, :5].detach().numpy()}")
    
    # 测试简化的单层双向RNN
    print("\n" + "=" * 50)
    print("测试简化单层双向RNN:")
    print("=" * 50)
    simple_rnn = SimpleBidirectionalRNN(input_dim, hidden_dim)
    outputs_simple, final_state_simple = simple_rnn(X)
    
    print(f"简化版输出形状: {outputs_simple.shape}")
    print(f"简化版最终状态形状: {final_state_simple.shape}")
    print(f"简化版输出[0,0,:5]: {outputs_simple[0, 0, :5].detach().numpy()}")
    print(f"简化版最终状态[0,:5]: {final_state_simple[0, :5].detach().numpy()}")
    
    # 测试多层手动实现（只测试num_layers=1，因为多层实现很复杂）
    print("\n" + "=" * 50)
    print("测试单层手动实现双向RNN:")
    print("=" * 50)
    manual_encoder_single = BidirectionalRNNEncoderManual(input_dim, hidden_dim, num_layers=1)
    outputs_manual_single, final_state_manual_single = manual_encoder_single(X)
    
    print(f"单层手动实现输出形状: {outputs_manual_single.shape}")
    print(f"单层手动实现最终状态形状: {final_state_manual_single.shape}")
    print(f"单层手动实现输出[0,0,:5]: {outputs_manual_single[0, 0, :5].detach().numpy()}")
    print(f"单层手动实现最终状态[0,:5]: {final_state_manual_single[0, :5].detach().numpy()}")
    
    # 测试LSTM和GRU
    print("\n" + "=" * 50)
    print("测试 LSTM 双向编码器:")
    print("=" * 50)
    encoder_lstm = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers, rnn_type='lstm')
    outputs_lstm, final_state_lstm = encoder_lstm(X)
    print(f"LSTM输出形状: {outputs_lstm.shape}")
    print(f"LSTM最终状态形状: {final_state_lstm.shape}")
    
    print("\n" + "=" * 50)
    print("测试 GRU 双向编码器:")
    print("=" * 50)
    encoder_gru = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers, rnn_type='gru')
    outputs_gru, final_state_gru = encoder_gru(X)
    print(f"GRU输出形状: {outputs_gru.shape}")
    print(f"GRU最终状态形状: {final_state_gru.shape}")

测试 PyTorch 内置双向RNN:
输入形状: torch.Size([5, 3, 10])
输出形状: torch.Size([5, 3, 16])
最终状态形状: torch.Size([3, 16])
输出[0,0,:5]: [-0.05503298  0.3152451  -0.12883385 -0.1644207   0.8629334 ]
最终状态[0,:5]: [-0.13639194 -0.70381755  0.6470107  -0.30530807 -0.56871176]

测试简化单层双向RNN:
简化版输出形状: torch.Size([5, 3, 16])
简化版最终状态形状: torch.Size([3, 16])
简化版输出[0,0,:5]: [-0.33399802 -0.4220133  -0.36363062 -0.6448077   0.63882095]
简化版最终状态[0,:5]: [ 0.4791242   0.40590534 -0.50094175  0.09135389 -0.10223829]

测试单层手动实现双向RNN:
单层手动实现输出形状: torch.Size([5, 3, 16])
单层手动实现最终状态形状: torch.Size([3, 16])
单层手动实现输出[0,0,:5]: [-0.29272994 -0.7368549  -0.83934057 -0.6380105   0.84614414]
单层手动实现最终状态[0,:5]: [-0.6238654  -0.24898647  0.37326643 -0.5239458  -0.19313827]

测试 LSTM 双向编码器:
LSTM输出形状: torch.Size([5, 3, 16])
LSTM最终状态形状: torch.Size([3, 16])

测试 GRU 双向编码器:
GRU输出形状: torch.Size([5, 3, 16])
GRU最终状态形状: torch.Size([3, 16])


Skip-gram模型中的负采样：

给定中心词 w_c 和上下文词 w_o，负采样损失函数为：

J = -log sigma(u_o^T * v_c) - sum_{k=1}^K log sigma(-u_{n_k}^T * v_c)

其中：
- sigma(x) = 1 / (1 + exp(-x)) 是sigmoid函数
- u_o 是上下文词 w_o 的词向量（输出向量）
- v_c 是中心词 w_c 的词向量（输入向量）
- u_{n_k} 是第 k 个负样本词的词向量
- K 是负样本数量

完整的目标函数（对数似然）：
L = log sigma(u_o^T * v_c) + sum_{k=1}^K log sigma(-u_{n_k}^T * v_c)

等价地，可以写成最大化形式：
L = log(1 / (1 + exp(-u_o^T * v_c))) + sum_{k=1}^K log(1 / (1 + exp(u_{n_k}^T * v_c)))

简化后：
L = -log(1 + exp(-u_o^T * v_c)) - sum_{k=1}^K log(1 + exp(u_{n_k}^T * v_c))

从噪声分布中采样负样本：
1. 噪声分布通常使用 unigram 分布，即根据词频进行采样。
2. 常用的方法是对词频进行某种变换，如 P(w) = (count(w))^(3/4) / Z，
   其中 Z 是归一化常数，3/4 次方可以平滑高频词的采样概率。
3. 负样本从训练数据中随机抽取，但不包括当前的正样本词对。
4. 每个样本抽取 K 个负样本（通常 K = 5-20）。

负采样方式的优点：
1. 计算效率高：只需要计算 K+1 个词的梯度，而不是整个词汇表。
2. 适用于大规模词汇表。
3. 能够学习到高质量的词向量。

In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class CBOWModel(nn.Module):
    """
    CBOW模型（使用完整softmax）
    """
    def __init__(self, vocab_size, embedding_dim):
        """
        参数：
            vocab_size: 词汇表大小 V
            embedding_dim: 嵌入维度 d
        """
        super(CBOWModel, self).__init__()
        
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # 输入权重矩阵 W: (V, d)
        self.W = nn.Parameter(torch.randn(vocab_size, embedding_dim) * 0.01)
        
        # 输出权重矩阵 W_out: (d, V)
        self.W_out = nn.Parameter(torch.randn(embedding_dim, vocab_size) * 0.01)
    
    def forward(self, context_indices, target_indices):
        """
        前向传播和损失计算
        
        参数：
            context_indices: 上下文词索引列表，形状 (batch_size, context_size)
            target_indices: 中心词索引列表，形状 (batch_size,)
        
        返回：
            loss: 交叉熵损失值
            logits: 输出概率分布的对数，形状 (batch_size, vocab_size)
        """
        batch_size, context_size = context_indices.shape
        
        # 1. 获取上下文词的嵌入向量
        # 从 W 中提取每个上下文词的向量
        # context_embeddings 形状: (batch_size, context_size, embedding_dim)
        context_embeddings = self.W[context_indices]  # (batch_size, context_size, d)
        
        # 2. 计算平均上下文向量作为隐藏层
        # 对 context_size 维度求平均
        hidden = torch.mean(context_embeddings, dim=1)  # (batch_size, d)
        
        # 3. 计算输出概率分布（logits）
        # logits = hidden * W_out
        logits = torch.matmul(hidden, self.W_out)  # (batch_size, vocab_size)
        
        # 4. 计算交叉熵损失
        # target_indices 形状: (batch_size,)
        loss = F.cross_entropy(logits, target_indices)
        
        return loss, logits


def cbow_forward_numpy(context_indices, target_indices, W, W_out):
    """
    使用NumPy实现的CBOW前向传播（用于理解底层计算）
    
    参数：
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        target_indices: 目标词索引，形状 (batch_size,)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回：
        loss: 交叉熵损失值
        logits: 输出概率分布的对数，形状 (batch_size, V)
    """
    batch_size, context_size = context_indices.shape
    
    # 1. 获取上下文词的嵌入向量
    context_embeddings = W[context_indices]  # (batch_size, context_size, d)
    
    # 2. 计算平均上下文向量
    hidden = np.mean(context_embeddings, axis=1)  # (batch_size, d)
    
    # 3. 计算logits
    logits = np.dot(hidden, W_out)  # (batch_size, V)
    
    # 4. 计算softmax概率
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    
    # 5. 计算交叉熵损失
    # loss = -sum(log(probs[i, target_indices[i]])) / batch_size
    loss = -np.mean(np.log(probs[np.arange(batch_size), target_indices] + 1e-8))
    
    return loss, logits


# 测试代码
def test_cbow():
    """
    测试CBOW模型的实现
    """
    # 参数设置
    vocab_size = 10
    embedding_dim = 5
    batch_size = 3
    context_size = 4
    
    print("=" * 60)
    print("测试 CBOW 模型")
    print("=" * 60)
    
    # 创建随机数据
    np.random.seed(42)
    torch.manual_seed(42)
    
    # 随机生成上下文词索引和目标词索引
    context_indices_np = np.random.randint(0, vocab_size, (batch_size, context_size))
    target_indices_np = np.random.randint(0, vocab_size, (batch_size,))
    
    # 转换为PyTorch张量
    context_indices = torch.tensor(context_indices_np, dtype=torch.long)
    target_indices = torch.tensor(target_indices_np, dtype=torch.long)
    
    print(f"上下文词索引 (batch_size={batch_size}, context_size={context_size}):")
    print(context_indices_np)
    print(f"\n目标词索引 (batch_size={batch_size}):")
    print(target_indices_np)
    
    # 使用PyTorch实现
    print("\n" + "-" * 40)
    print("PyTorch 实现:")
    print("-" * 40)
    
    model = CBOWModel(vocab_size, embedding_dim)
    loss, logits = model(context_indices, target_indices)
    
    print(f"损失值: {loss.item():.6f}")
    print(f"Logits 形状: {logits.shape}")
    print(f"Logits 前5个值 (第一个样本): {logits[0, :5].detach().numpy()}")
    
    # 使用NumPy实现（使用相同的权重）
    print("\n" + "-" * 40)
    print("NumPy 实现:")
    print("-" * 40)
    
    # 获取模型的权重并转换为NumPy
    W_np = model.W.detach().numpy()
    W_out_np = model.W_out.detach().numpy()
    
    loss_np, logits_np = cbow_forward_numpy(
        context_indices_np, target_indices_np, W_np, W_out_np
    )
    
    print(f"损失值: {loss_np:.6f}")
    print(f"Logits 形状: {logits_np.shape}")
    print(f"Logits 前5个值 (第一个样本): {logits_np[0, :5]}")
    
    # 验证两种实现是否一致
    print("\n" + "-" * 40)
    print("验证结果:")
    print("-" * 40)
    loss_diff = abs(loss.item() - loss_np)
    logits_diff = np.max(np.abs(logits.detach().numpy() - logits_np))
    
    print(f"损失差异: {loss_diff:.10f}")
    print(f"Logits 最大差异: {logits_diff:.10f}")
    
    if loss_diff < 1e-6 and logits_diff < 1e-6:
        print("✓ 两种实现结果一致！")
    else:
        print("✗ 两种实现结果存在差异，请检查实现。")


# 展示CBOW模型的详细工作流程
def demonstrate_cbow_workflow():
    """
    详细展示CBOW模型的工作流程
    """
    print("\n" + "=" * 60)
    print("CBOW 模型详细工作流程演示")
    print("=" * 60)
    
    # 简化示例
    vocab_size = 5
    embedding_dim = 3
    context_size = 3
    
    print(f"词汇表大小: {vocab_size}")
    print(f"嵌入维度: {embedding_dim}")
    print(f"上下文窗口大小: {context_size}")
    print("\n词汇表示例: ['apple', 'banana', 'orange', 'grape', 'watermelon']")
    
    # 创建权重矩阵
    W = np.array([
        [0.1, 0.2, 0.3],   # apple
        [0.4, 0.5, 0.6],   # banana
        [0.7, 0.8, 0.9],   # orange
        [1.0, 1.1, 1.2],   # grape
        [1.3, 1.4, 1.5]    # watermelon
    ])
    
    W_out = np.array([
        [0.1, 0.2, 0.3, 0.4, 0.5],
        [0.6, 0.7, 0.8, 0.9, 1.0],
        [1.1, 1.2, 1.3, 1.4, 1.5]
    ])
    
    print(f"\n输入权重矩阵 W 形状: {W.shape}")
    print(f"输出权重矩阵 W_out 形状: {W_out.shape}")
    
    # 示例样本：中心词是 'orange' (索引2)，上下文词是 ['apple', 'banana', 'grape'] (索引0, 1, 3)
    context_indices = np.array([[0, 1, 3]])  # ['apple', 'banana', 'grape']
    target_indices = np.array([2])          # 'orange'
    
    print(f"\n样本数据:")
    print(f"上下文词索引: {context_indices[0]} (['apple', 'banana', 'grape'])")
    print(f"目标词索引: {target_indices[0]} ('orange')")
    
    # 1. 获取上下文词嵌入
    context_embeddings = W[context_indices]
    print(f"\n1. 上下文词嵌入向量 (形状: {context_embeddings.shape}):")
    for i, idx in enumerate(context_indices[0]):
        print(f"  词 '{['apple','banana','orange','grape','watermelon'][idx]}' (索引{idx}): {context_embeddings[0, i]}")
    
    # 2. 计算平均上下文向量
    hidden = np.mean(context_embeddings, axis=1)
    print(f"\n2. 平均上下文向量 (形状: {hidden.shape}):")
    print(f"   {hidden[0]}")
    
    # 3. 计算logits
    logits = np.dot(hidden, W_out)
    print(f"\n3. Logits (形状: {logits.shape}):")
    print(f"   {logits[0]}")
    
    # 4. 计算softmax概率
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    print(f"\n4. Softmax概率分布:")
    for i, prob in enumerate(probs[0]):
        print(f"   P('{['apple','banana','orange','grape','watermelon'][i]}') = {prob:.4f}")
    
    # 5. 计算损失
    loss = -np.log(probs[0, target_indices[0]])
    print(f"\n5. 交叉熵损失 (目标词 'orange'):")
    print(f"   损失值: {loss:.4f}")
    
    print("\n" + "=" * 60)
    print("CBOW 模型总结:")
    print("=" * 60)
    print("1. 输入: 上下文词索引列表")
    print("2. 嵌入层: 将每个上下文词映射为向量")
    print("3. 隐藏层: 计算上下文词向量的平均值")
    print("4. 输出层: 通过输出权重矩阵计算logits")
    print("5. Softmax: 转换为概率分布")
    print("6. 损失: 使用交叉熵损失，目标为中心词")


if __name__ == "__main__":
    # 运行测试
    test_cbow()
    
    # 展示详细工作流程
    demonstrate_cbow_workflow()

测试 CBOW 模型
上下文词索引 (batch_size=3, context_size=4):
[[6 3 7 4]
 [6 9 2 6]
 [7 4 3 7]]

目标词索引 (batch_size=3):
[7 2 5]

----------------------------------------
PyTorch 实现:
----------------------------------------
损失值: 2.302581
Logits 形状: torch.Size([3, 10])
Logits 前5个值 (第一个样本): [ 1.0202645e-04 -7.0082671e-05 -1.4082417e-04 -5.8724127e-05
  2.0844302e-04]

----------------------------------------
NumPy 实现:
----------------------------------------
损失值: 2.302581
Logits 形状: (3, 10)
Logits 前5个值 (第一个样本): [ 1.0202645e-04 -7.0082671e-05 -1.4082417e-04 -5.8724127e-05
  2.0844302e-04]

----------------------------------------
验证结果:
----------------------------------------
损失差异: 0.0000000000
Logits 最大差异: 0.0000000000
✓ 两种实现结果一致！

CBOW 模型详细工作流程演示
词汇表大小: 5
嵌入维度: 3
上下文窗口大小: 3

词汇表示例: ['apple', 'banana', 'orange', 'grape', 'watermelon']

输入权重矩阵 W 形状: (5, 3)
输出权重矩阵 W_out 形状: (3, 5)

样本数据:
上下文词索引: [0 1 3] (['apple', 'banana', 'grape'])
目标词索引: 2 ('orange')

1. 上下文词嵌入向量 (形状: (1, 3, 3)):
  词 'apple' (索引0): [

给定：
Q ∈ R^(2×4), K ∈ R^(3×4), V ∈ R^(3×5)
d_k = 4

步骤1：计算得分矩阵
S = Q * K^T / sqrt(d_k)

设 Q = [[q11, q12, q13, q14],
        [q21, q22, q23, q24]]

K = [[k11, k12, k13, k14],
     [k21, k22, k23, k24],
     [k31, k32, k33, k34]]

K^T = [[k11, k21, k31],
       [k12, k22, k32],
       [k13, k23, k33],
       [k14, k24, k34]]

Q * K^T = [[q11*k11+q12*k12+q13*k13+q14*k14, q11*k21+q12*k22+q13*k23+q14*k24, q11*k31+q12*k32+q13*k33+q14*k34],
           [q21*k11+q22*k12+q23*k13+q24*k14, q21*k21+q22*k22+q23*k23+q24*k24, q21*k31+q22*k32+q23*k33+q24*k34]]

S = (1/sqrt(4)) * (Q * K^T) = (1/2) * (Q * K^T)

因此 S 的形状为 (2, 3)

步骤2：对得分矩阵应用 Softmax（按行）
对于每一行 i，计算：
softmax(S_i) = exp(S_i) / sum(exp(S_j))

得到注意力权重矩阵 A，形状为 (2, 3)
A[i, j] = exp(S[i, j]) / sum_{k=1}^3 exp(S[i, k])

步骤3：加权求和得到输出
Output = A * V

Output[i, :] = sum_{j=1}^3 A[i, j] * V[j, :]

Output 的形状为 (2, 5)

具体数值示例：
假设：
Q = [[1, 0, 1, 0],
     [0, 1, 0, 1]]

K = [[1, 1, 1, 1],
     [2, 2, 2, 2],
     [3, 3, 3, 3]]

V = [[1, 2, 3, 4, 5],
     [6, 7, 8, 9, 10],
     [11, 12, 13, 14, 15]]

计算 Q*K^T：
Q*K^T = [[1*1+0*1+1*1+0*1, 1*2+0*2+1*2+0*2, 1*3+0*3+1*3+0*3],
         [0*1+1*1+0*1+1*1, 0*2+1*2+0*2+1*2, 0*3+1*3+0*3+1*3]]
      = [[2, 4, 6],
         [2, 4, 6]]

S = (1/2) * [[2, 4, 6],
             [2, 4, 6]]
  = [[1, 2, 3],
     [1, 2, 3]]

对每行应用 Softmax：
行1：exp(1)=2.718, exp(2)=7.389, exp(3)=20.085
sum = 30.192
A[0,:] = [0.0900, 0.2447, 0.6653]

行2：同样
A[1,:] = [0.0900, 0.2447, 0.6653]

计算输出：
Output[0,:] = 0.0900*[1,2,3,4,5] + 0.2447*[6,7,8,9,10] + 0.6653*[11,12,13,14,15]
           = [0.0900+1.4682+7.3183, 0.1800+1.7129+7.9836, 0.2700+1.9576+8.6489, 0.3600+2.2023+9.3142, 0.4500+2.4470+9.9795]
           = [8.8765, 9.8765, 10.8765, 11.8765, 12.8765]

Output[1,:] = 相同（因为两行相同）
           = [8.8765, 9.8765, 10.8765, 11.8765, 12.8765]

最终输出矩阵形状为 (2, 5)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    """
    def __init__(self, d_model, num_heads):
        """
        参数：
            d_model: 模型维度
            num_heads: 注意力头数
        """
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        缩放点积注意力
        
        参数：
            Q: 查询矩阵，形状 (batch, num_heads, seq_len, d_k)
            K: 键矩阵，形状 (batch, num_heads, seq_len, d_k)
            V: 值矩阵，形状 (batch, num_heads, seq_len, d_v)
            mask: 掩码矩阵，可选
        
        返回：
            output: 注意力输出，形状 (batch, num_heads, seq_len, d_v)
            attention_weights: 注意力权重，形状 (batch, num_heads, seq_len, seq_len)
        """
        # 计算得分矩阵
        # Q * K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # scores 形状: (batch, num_heads, seq_len, seq_len)
        
        # 如果有掩码，应用掩码
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Softmax
        attention_weights = F.softmax(scores, dim=-1)
        # attention_weights 形状: (batch, num_heads, seq_len, seq_len)
        
        # 加权求和
        output = torch.matmul(attention_weights, V)
        # output 形状: (batch, num_heads, seq_len, d_v)
        
        return output, attention_weights
    
    def forward(self, X, mask=None):
        """
        前向传播
        
        参数：
            X: 输入序列，形状 (seq_len, batch, d_model)
            mask: 掩码矩阵，可选
        
        返回：
            output: 注意力输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 2. 重塑为多头形式
        # 将 (seq_len, batch, d_model) 转换为 (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v)
        
        # 转置为 (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
        K = K.permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
        V = V.permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_v)
        
        # 3. 计算缩放点积注意力
        attn_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        # attn_output 形状: (batch, num_heads, seq_len, d_v)
        
        # 4. 拼接所有头的输出
        # 转置回 (seq_len, batch, num_heads, d_v)
        attn_output = attn_output.permute(2, 0, 1, 3)  # (seq_len, batch, num_heads, d_v)
        
        # 重塑为 (seq_len, batch, d_model)
        attn_output = attn_output.contiguous().view(seq_len, batch_size, self.d_model)
        # attn_output 形状: (seq_len, batch, d_model)
        
        # 5. 最终线性层
        output = self.W_o(attn_output)  # (seq_len, batch, d_model)
        
        return output, attention_weights


# 简化版多头注意力（不使用线性层，直接使用随机投影）
class SimpleMultiHeadAttention(nn.Module):
    """
    简化版多头注意力（用于演示）
    """
    def __init__(self, d_model, num_heads):
        super(SimpleMultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
    def forward(self, X):
        """
        前向传播（使用随机投影）
        """
        seq_len, batch_size, d_model = X.shape
        
        # 随机线性投影（仅用于演示）
        W_q = torch.randn(d_model, d_model)
        W_k = torch.randn(d_model, d_model)
        W_v = torch.randn(d_model, d_model)
        W_o = torch.randn(d_model, d_model)
        
        # 计算 Q, K, V
        Q = torch.matmul(X, W_q)  # (seq_len, batch, d_model)
        K = torch.matmul(X, W_k)
        V = torch.matmul(X, W_v)
        
        # 重塑为多头
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v).permute(1, 2, 0, 3)
        
        # 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attention_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attention_weights, V)
        
        # 拼接
        attn_output = attn_output.permute(2, 0, 1, 3).contiguous()
        attn_output = attn_output.view(seq_len, batch_size, d_model)
        
        # 最终线性层
        output = torch.matmul(attn_output, W_o)
        
        return output, attention_weights


# 测试代码
def test_multi_head_attention():
    """
    测试多头注意力
    """
    print("=" * 60)
    print("测试多头注意力机制")
    print("=" * 60)
    
    # 参数设置
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch_size = 2
    
    print(f"d_model = {d_model}")
    print(f"num_heads = {num_heads}")
    print(f"每个头的维度 d_k = d_v = {d_model // num_heads}")
    print(f"序列长度 = {seq_len}")
    print(f"批次大小 = {batch_size}")
    
    # 创建随机输入
    torch.manual_seed(42)
    X = torch.randn(seq_len, batch_size, d_model)
    
    print(f"\n输入 X 形状: {X.shape}")
    print(f"输入 X (第一个时间步, 第一个样本): {X[0, 0, :].detach().numpy()}")
    
    # 创建多头注意力模型
    model = MultiHeadAttention(d_model, num_heads)
    
    # 前向传播
    output, attention_weights = model(X)
    
    print(f"\n输出形状: {output.shape}")
    print(f"输出 (第一个时间步, 第一个样本): {output[0, 0, :].detach().numpy()}")
    print(f"注意力权重形状: {attention_weights.shape}")
    
    # 展示多头注意力的详细计算过程
    print("\n" + "=" * 60)
    print("多头注意力详细计算过程演示")
    print("=" * 60)
    
    # 简化的数值示例
    d_model = 4
    num_heads = 2
    d_k = d_model // num_heads  # 2
    seq_len = 2
    batch_size = 1
    
    print(f"\n使用简化示例:")
    print(f"d_model = {d_model}, num_heads = {num_heads}, d_k = {d_k}")
    print(f"seq_len = {seq_len}, batch_size = {batch_size}")
    
    # 创建简单的输入
    X_simple = torch.tensor([[[1.0, 2.0, 3.0, 4.0]],
                             [[5.0, 6.0, 7.0, 8.0]]])
    print(f"\n输入 X (seq_len={seq_len}, batch={batch_size}, d_model={d_model}):")
    print(X_simple.squeeze().numpy())
    
    # 使用简化的线性投影（单位矩阵）
    class TestMultiHeadAttention(MultiHeadAttention):
        def __init__(self, d_model, num_heads):
            super().__init__(d_model, num_heads)
            # 使用单位矩阵作为投影权重（仅用于演示）
            self.W_q.weight.data = torch.eye(d_model)
            self.W_k.weight.data = torch.eye(d_model)
            self.W_v.weight.data = torch.eye(d_model)
            self.W_o.weight.data = torch.eye(d_model)
    
    model_test = TestMultiHeadAttention(d_model, num_heads)
    output_test, attn_weights_test = model_test(X_simple)
    
    print(f"\n输出 (seq_len={seq_len}, batch={batch_size}, d_model={d_model}):")
    print(output_test.squeeze().detach().numpy())
    print(f"\n注意力权重形状: {attn_weights_test.shape}")
    print(f"注意力权重 (batch=0, head=0):")
    print(attn_weights_test[0, 0].detach().numpy())
    print(f"\n注意力权重 (batch=0, head=1):")
    print(attn_weights_test[0, 1].detach().numpy())


# 展示多头注意力的工作流程
def demonstrate_attention_flow():
    """
    展示多头注意力的工作流程
    """
    print("\n" + "=" * 60)
    print("多头注意力工作流程")
    print("=" * 60)
    
    d_model = 4
    num_heads = 2
    d_k = d_model // num_heads
    
    print(f"1. 输入: X ∈ R^(seq_len × batch × d_model), d_model={d_model}")
    print(f"2. 线性投影: Q, K, V = X * W_q, X * W_k, X * W_v")
    print(f"3. 重塑为多头: Q, K, V 形状变为 (batch, num_heads, seq_len, d_k)")
    print(f"   - num_heads={num_heads}, d_k={d_k}")
    print(f"4. 对每个头计算缩放点积注意力:")
    print(f"   - scores = Q * K^T / sqrt(d_k)")
    print(f"   - attention_weights = softmax(scores)")
    print(f"   - head_output = attention_weights * V")
    print(f"5. 拼接所有头的输出: (seq_len, batch, num_heads * d_v)")
    print(f"6. 最终线性投影: output = concat * W_o")
    print(f"7. 输出: 与输入形状相同 (seq_len, batch, d_model)")
    
    print(f"\n优点:")
    print(f"- 不同的头可以关注不同的特征子空间")
    print(f"- 提高了模型的表达能力")
    print(f"- 能够捕获多种依赖关系")


if __name__ == "__main__":
    # 测试多头注意力
    test_multi_head_attention()
    
    # 展示工作流程
    demonstrate_attention_flow()

测试多头注意力机制
d_model = 4
num_heads = 2
每个头的维度 d_k = d_v = 2
序列长度 = 3
批次大小 = 2

输入 X 形状: torch.Size([3, 2, 4])
输入 X (第一个时间步, 第一个样本): [ 1.9269153  1.4872841  0.9007172 -2.105521 ]

输出形状: torch.Size([3, 2, 4])
输出 (第一个时间步, 第一个样本): [-0.23971008 -0.20133398 -0.11523722 -0.40986055]
注意力权重形状: torch.Size([2, 2, 3, 3])

多头注意力详细计算过程演示

使用简化示例:
d_model = 4, num_heads = 2, d_k = 2
seq_len = 2, batch_size = 1

输入 X (seq_len=2, batch=1, d_model=4):
[[1. 2. 3. 4.]
 [5. 6. 7. 8.]]

输出 (seq_len=2, batch=1, d_model=4):
[[4.9991746 5.9991746 7.        8.       ]
 [5.        6.        7.        8.       ]]

注意力权重形状: torch.Size([1, 2, 2, 2])
注意力权重 (batch=0, head=0):
[[2.0644255e-04 9.9979359e-01]
 [3.0755807e-14 1.0000000e+00]]

注意力权重 (batch=0, head=1):
[[2.5200373e-09 1.0000000e+00]
 [3.7535794e-19 1.0000000e+00]]

多头注意力工作流程
1. 输入: X ∈ R^(seq_len × batch × d_model), d_model=4
2. 线性投影: Q, K, V = X * W_q, X * W_k, X * W_v
3. 重塑为多头: Q, K, V 形状变为 (batch, num_heads, seq_len, d_k)
   - num_heads=2, d_k=2
4. 对每个头计算缩